# Mass Spectrometry Analysis of Extracellular Vesicles Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR²](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, referencing all entities by their `@id` fields.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review all available Record Sets, Fields, and Columns. All references use the entity `@id` as specified by the schema.

In [ ]:
from mlcroissant._dataset.structure_metadata import RecordSet, Field

# List all record sets and their fields by @id
print("Available RecordSets:")
record_sets = []
for record_set in metadata.record_sets:
    print(f"- RecordSet @id: {record_set.id}")
    print(f"    Name: {record_set.name}")
    print(f"    Description: {getattr(record_set, 'description', '')}")
    # List fields
    print("    Fields:")
    for field in record_set.fields:
        print(f"      - Field @id: {field.id}, Name: {field.name}, DataType: {field.data_type}")
    record_sets.append(record_set.id)

if not record_sets:
    print("No record sets found. Please check the dataset schema and updates.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the RecordSet and Field `@id` values discovered above.

In [ ]:
# Re-run the data overview code above if you need to see RecordSet and Field @id values.

# For the further steps, pick the first RecordSet by @id
if len(record_sets) == 0:
    raise RuntimeError("No record sets are available in this dataset.")

dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display columns for the first record set
first_rs = record_sets[0]
print(f"Columns in RecordSet {first_rs}:")
print(dataframes[first_rs].columns.tolist())

# Preview the records
dataframes[first_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply standard EDA: filtering, normalization, and grouping. Use Fields by their `@id`.

In [ ]:
# --- Select a numeric field by its @id from the Data Overview section ---
# Edit these values as appropriate for your dataset

record_set_id = first_rs   # The @id of the RecordSet to analyze

# Choose a numeric field @id (example: 'cr:coverage')
numeric_field = None
group_field = None

# Search for numeric field (try to use the first Float/Number/Integer field)
for record_set in metadata.record_sets:
    if record_set.id == record_set_id:
        for field in record_set.fields:
            if field.data_type in ['Float', 'Number', 'Integer', 'schema:Float', 'schema:Number', 'schema:Integer']:
                if field.id in dataframes[record_set_id].columns:
                    numeric_field = field.id
                    break
        # Also find a possible group field (for demonstration, use first non-numeric field)
        for field in record_set.fields:
            if field.data_type not in ['Float', 'Number', 'Integer', 'schema:Float', 'schema:Number', 'schema:Integer']:
                if field.id in dataframes[record_set_id].columns:
                    group_field = field.id
                    break
        break

if numeric_field is None:
    raise RuntimeError("No numeric field found in this record set. Please adjust your field selection.")
if group_field is None:
    print("No group field found, grouping step will be skipped.")

df = dataframes[record_set_id]

# Filter by threshold (use 10 as a demonstration, adjust as appropriate)
threshold = 10
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold} (showing top 5):")
print(filtered_df.head())

# Normalize numeric field
norm_col = f"{numeric_field}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} (top 5 rows):")
print(filtered_df[[numeric_field, norm_col]].head())

# Group by group_field if found
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nMean {numeric_field} grouped by {group_field} (top 5):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship with the group field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Histogram of numeric field
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field} (filtered > {threshold})")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot by group (if group field exists)
if group_field and group_field in filtered_df.columns:
    plt.figure(figsize=(12,5))
    sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
    plt.title(f"{numeric_field} grouped by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded mass spectrometry data using the FAIR² Croissant schema and explored its structure and contents by referencing all entities via their `@id`. We filtered, normalized, grouped, and visualized key attributes. For advanced analytics or modeling, see the [`mlcroissant` documentation](https://croissant.docs.mlcommons.org/) and the field definitions in the dataset metadata.

Adjust the field and group selection above (see section 2 for `@id`s) to reflect your analytic goals.